# LIBERO — **eval_info 누락 분석 + 로그에서 SR 복구**

eval 은 돌았고(action 저장됨) 결과는 **로그에만** 있고 `eval_info.json` 이 안 남은 경우를 찾아 복구한다.

- eval 은 rollout 중에 action(.pt)을 저장하고, **맨 끝에** `eval_info.json` 을 쓴다.
  중간에 죽으면 action 은 있는데 eval_info 는 없다 → SR 은 로그의 `Aggregated Metrics for overall` 에 남음.
- ① 인벤토리(action/eval_info/log) → ② 로그에서 SR 복구 → ③ (선택) eval_info.json 복원.
- ⚠️ LIBERO-10 = 10 task → **overall n_ep = per-task × 10**. per-task 500 = 정상, 50 = 옛 eval.


In [ ]:
import sys, json, re
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)

TASK   = 'libero_10'
MODELS = ['act', 'acm', 'acm2', 'mosaic', 'bimamba', 'bimamba_s7']
SEEDS  = [0, 1, 2, 3]
N_TASKS = 10                              # LIBERO-10 = 10개 task. per-task 500 → overall 5000
EVAL_ROOT = cf.OUTPUT_BASE / 'eval_clean' / TASK
LOG_DIR   = cf.OUTPUT_BASE / '_logs'
print('eval:', EVAL_ROOT, '| exists:', EVAL_ROOT.is_dir())
print('logs:', LOG_DIR, '| exists:', LOG_DIR.is_dir(),
      '| .log 개수:', len(list(LOG_DIR.glob('*.log'))) if LOG_DIR.is_dir() else 0)

## 1) 인벤토리 — action 개수 / eval_info 유무 / 매칭 로그


In [ ]:
# ── 인벤토리: (모델,seed) 별 action 개수 / eval_info 유무 / 매칭 로그 ──
def action_dirs(tag, seed):
    d = EVAL_ROOT / tag / f'seed{seed}'
    return sorted(d.rglob('action_logs')) if d.is_dir() else []

def n_actions(tag, seed):
    return sum(len(list(al.glob('episode_*.pt'))) for al in action_dirs(tag, seed))

def info_files(tag, seed):
    d = EVAL_ROOT / tag / f'seed{seed}'
    return sorted(d.rglob('eval_info.json')) if d.is_dir() else []

def logs_for(tag, seed):
    if not LOG_DIR.is_dir():
        return []
    key = f'{tag}__seed{seed}'                 # 로그 파일명 규칙: ..{tag}__seed{N}..
    return sorted(p for p in LOG_DIR.glob('*.log') if key in p.name)

print(f"{'model':<12}{'seed':>5}{'#action':>9}{'eval_info':>11}{'#log':>6}")
print('-' * 45)
missing = []                                    # action 있는데 eval_info 없는 것
for tag in MODELS:
    for s in SEEDS:
        na = n_actions(tag, s)
        infos = info_files(tag, s)
        lg = logs_for(tag, s)
        flag = 'O' if infos else ('❌없음' if na > 0 else '-')
        print(f'{tag:<12}{s:>5}{na:>9}{flag:>11}{len(lg):>6}')
        if na > 0 and not infos:
            missing.append((tag, s, na, lg))
print('\n■ action 있는데 eval_info 없는 (모델,seed):',
      [f'{t}/s{s}({na}개, log {len(lg)})' for t, s, na, lg in missing] or '없음')

## 2) 로그에서 SR 복구 (eval_info 없는 것)


In [ ]:
# ── 로그에서 SR 복구: 'Aggregated Metrics for overall' 블록의 pc_success / n_episodes ──
def parse_overall(text):
    # eval 은 json 저장 직전에 overall dict 를 print 한다. 그 블록을 우선 찾는다.
    m = re.search(r'Aggregated Metrics for overall', text)
    seg = text[m.start(): m.start() + 800] if m else text
    pc = re.findall(r"pc_success['\"]?\s*[:=]\s*([0-9]+\.?[0-9]*)", seg)
    ne = re.findall(r"n_episodes['\"]?\s*[:=]\s*([0-9]+)", seg)
    if not m:
        # overall 마커가 없으면: 전체에서 n_episodes 가 가장 큰(=overall) 쌍을 고른다
        allpc = re.findall(r"pc_success['\"]?\s*[:=]\s*([0-9]+\.?[0-9]*)", text)
        allne = re.findall(r"n_episodes['\"]?\s*[:=]\s*([0-9]+)", text)
        if allne:
            j = max(range(len(allne)), key=lambda i: int(allne[i]))
            return (float(allpc[j]) if j < len(allpc) else None, int(allne[j]))
        return (None, None)
    return (float(pc[0]) if pc else None, int(ne[0]) if ne else None)

print('로그에서 복구한 결과 (eval_info 없는 것):\n')
recovered = []                                  # (tag, seed, sr, n_ep_overall, per_task, logpath, out_dir)
for tag, s, na, lg in missing:
    best = None
    for lp in lg:
        sr, ne = parse_overall(lp.read_text(errors='ignore'))
        if sr is not None and (best is None or (ne or 0) > (best[1] or 0)):
            best = (sr, ne, lp)
    if best is None:
        print(f'  {tag}/seed{s}: 로그에서 SR 못 찾음 (eval 미완/로그 없음) — 재eval 필요')
        continue
    sr, ne, lp = best
    per_task = (ne // N_TASKS) if ne else None
    # eval_info 를 쓸 위치 = action_logs 의 부모의 부모 (…/rep0/actions/action_logs → …/rep0)
    al = action_dirs(tag, s)[0]
    out_dir = al.parent.parent                  # rep0 (또는 seed 디렉토리)
    recovered.append((tag, s, sr, ne, per_task, lp, out_dir))
    print(f'  {tag}/seed{s}: SR={sr:.1f}%  overall n_ep={ne} (per-task {per_task})  ← {lp.name}')
print('\n→ per-task 500 이면 정상 500ep eval (overall 5000). 50 이면 옛 50ep.')

## 3) (선택) eval_info.json 복원 — dry-run → EXECUTE=True


In [ ]:
# ── (선택) eval_info.json 복원: 로그에서 읽은 값으로 최소 eval_info 생성 ── EXECUTE=True ──
#    eval_final/eval_node 가 이걸 읽어 SR 을 집계한다. 원본 로그 값 그대로 기록.
EXECUTE = False

if not recovered:
    print('복원할 것 없음.')
else:
    for tag, s, sr, ne, per_task, lp, out_dir in recovered:
        target = out_dir / 'eval_info.json'
        payload = {'overall': {'pc_success': sr, 'n_episodes': ne},
                   '_recovered_from_log': str(lp.name)}
        if target.exists():
            print(f'  skip (이미 있음): {target}')
            continue
        print(f'  {"쓴다" if EXECUTE else "DRY-RUN"}: {target}  (SR={sr:.1f}%, n_ep={ne})')
        if EXECUTE:
            target.write_text(json.dumps(payload, indent=2))
    print('\n' + ('복원 완료 → 이제 eval_final 재실행하면 반영됨.' if EXECUTE
                   else '확인됐으면 EXECUTE=True 로 다시 실행.'))